In [1]:
!pip install --no-cache-dir -q \
--trusted-host pypi.org \
--trusted-host files.pythonhosted.org \
--trusted-host pypi.python.org \
timm


[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python -m pip install --upgrade pip


In [ ]:
from huggingface_hub import login
from huggingface_hub import whoami
login(token="hf_key")
print(f"✓ {whoami()['name']}")

✓ Ranjan637


In [15]:
from huggingface_hub import hf_hub_download
def download_all_chunks(repo_id, raw_root, train_parts=5, val_parts=5):
    os.makedirs(raw_root, exist_ok=True)
    for split, n in [("train", train_parts), ("val", val_parts)]:
        for p in range(1, n + 1):
            fn = f"imagenet100_{split}_part{p}.zip"
            print(f"↓ {fn}")
            hf_hub_download(repo_id=repo_id, filename=fn,
                            repo_type="dataset", local_dir=raw_root)
    hf_hub_download(repo_id=repo_id, filename="Labels.json",
                    repo_type="dataset", local_dir=raw_root)
    print("Done.")

download_all_chunks(HF_REPO, RAW_ROOT)

↓ imagenet100_train_part1.zip
↓ imagenet100_train_part2.zip
↓ imagenet100_train_part3.zip
↓ imagenet100_train_part4.zip
↓ imagenet100_train_part5.zip
↓ imagenet100_val_part1.zip
↓ imagenet100_val_part2.zip
↓ imagenet100_val_part3.zip
↓ imagenet100_val_part4.zip
↓ imagenet100_val_part5.zip
Done.


In [16]:
import os, zipfile, shutil
# =====================================================================
# Cell 5 — Extract + merge
# =====================================================================
def extract_and_merge(raw_root, data_root, train_parts=5, val_parts=5):
    tmp = os.path.join(raw_root, "_extract_tmp")
    os.makedirs(tmp, exist_ok=True)
    for split, n in [("train", train_parts), ("val", val_parts)]:
        split_dir = os.path.join(data_root, split)
        os.makedirs(split_dir, exist_ok=True)
        for p in range(1, n + 1):
            zp = os.path.join(raw_root, f"imagenet100_{split}_part{p}.zip")
            if not os.path.exists(zp): continue
            et = os.path.join(tmp, f"{split}_{p}")
            os.makedirs(et, exist_ok=True)
            with zipfile.ZipFile(zp) as zf:
                zf.extractall(et)
            for root, _, files in os.walk(et):
                if any(f.lower().endswith((".jpg",".jpeg",".png")) for f in files):
                    cls = os.path.basename(root)
                    dest = os.path.join(split_dir, cls)
                    os.makedirs(dest, exist_ok=True)
                    for f in files:
                        if f.lower().endswith((".jpg",".jpeg",".png")):
                            d = os.path.join(dest, f)
                            if not os.path.exists(d):
                                shutil.move(os.path.join(root, f), d)
    shutil.rmtree(tmp, ignore_errors=True)
    print("Done.")

extract_and_merge(RAW_ROOT, DATA_ROOT)

Done.


In [33]:
# =====================================================================
# Cell 1 — Phase 2 config
# =====================================================================
import os, time, math, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.amp import GradScaler, autocast
from timm.scheduler import CosineLRScheduler
from tqdm.auto import tqdm

PHASE2_DIR = "./outputs/phase2"
os.makedirs(PHASE2_DIR, exist_ok=True)

P2_EPOCHS = 30
P2_BLOCKS = 12
P2_K      = 9
P2_BATCH  = 128
P2_LR     = 1e-3
P2_WARMUP = 5
P2_WD     = 0.05
GRAD_CLIP = 1.0

# Baseline reference (mean of P1-12 and P5-k9), with ±noise floor
BASELINE_MEAN = 0.4520
BASELINE_STD  = 0.0040   # ±0.4% empirical noise floor from Phase 1

print(f"Phase 2 config: {P2_BLOCKS} blocks, k={P2_K}, {P2_EPOCHS} epochs")
print(f"Baseline reference: {BASELINE_MEAN:.4f} ± {BASELINE_STD:.4f}")

Phase 2 config: 12 blocks, k=9, 30 epochs
Baseline reference: 0.4520 ± 0.0040


In [34]:
# =====================================================================
# Cell 2 (REVISED) — DiffGroupNorm with corrected batch variant
# =====================================================================
class DiffGroupNorm(nn.Module):
    """
    Two variants of group-style normalization for ViG.

    variant='per_image':
        GroupNorm over (C, N) per image. Groups partition channels;
        normalization is across channels-within-group AND across N patches.
        This is the variant that beat the baseline in P6a.

    variant='batch':
        BatchNorm1d over the (B*N) rows, per channel. Normalizes each
        channel using batch statistics across all patches and images.
        Preserves per-image DC information (does not center over N).
        NOTE: this is NOT GroupNorm — GroupNorm cannot do per-channel
        normalization across the batch axis.
    """
    def __init__(self, ch, groups=8, variant='per_image', eps=1e-6):
        super().__init__()
        self.variant = variant
        self.ch = ch
        if variant == 'per_image':
            self.norm = nn.GroupNorm(groups, ch, eps=eps)
        elif variant == 'batch':
            self.norm = nn.BatchNorm1d(ch, eps=eps, momentum=0.1)
        else:
            raise ValueError(f"unknown variant: {variant}")

    def forward(self, x_flat, B, N):
        C = x_flat.size(-1)
        if self.variant == 'per_image':
            # (B*N, C) -> (B, C, N) -> GroupNorm over (C_group, N)
            x = x_flat.view(B, N, C).permute(0, 2, 1)   # (B, C, N)
            x = self.norm(x)
            return x.permute(0, 2, 1).reshape(B * N, C)
        else:
            # batch: normalizes per-channel over the (B*N) axis
            return self.norm(x_flat)

In [35]:
# =====================================================================
# Cell 3 — P7: ScaledResidual for the GCN branch
# =====================================================================
class ScaledResidual(nn.Module):
    """x + alpha * sublayer_out, alpha learnable per layer."""
    def __init__(self, init_alpha=0.3):
        super().__init__()
        self.alpha = nn.Parameter(torch.tensor(float(init_alpha)))

    def forward(self, x, sublayer_out):
        return x + self.alpha * sublayer_out

In [36]:
# =====================================================================
# Cell 4 (FIXED) — GrapherP2 / PyramidViGP2 / build_p2_model
# =====================================================================
class GrapherP2(nn.Module):
    """
    Grapher with optional anti-smoothing components:
      p6_mode: None | 'per_image' | 'batch'   (DiffGroupNorm variant)
      p7_mode: None | 'scaled'                (ScaledResidual on GCN)
    """
    def __init__(self, ch, k=9, drop_path=0.0,
                 p6_mode=None, p6_groups=8,
                 p7_mode=None, p7_alpha=0.3):
        super().__init__()
        self.k, self.ch = k, ch

        self.norm1 = LayerNorm2d(ch)
        self.gcn   = MRConv2d(ch, ch)
        self.norm2 = LayerNorm2d(ch)
        self.ffn = nn.Sequential(
            nn.Conv2d(ch, ch * 4, 1, bias=False),
            nn.GELU(),
            nn.Conv2d(ch * 4, ch, 1, bias=False),
        )
        self.drop = DropPath(drop_path) if drop_path > 0 else nn.Identity()

        # P6: DiffGroupNorm after GCN
        self.p6 = DiffGroupNorm(ch, groups=p6_groups, variant=p6_mode) \
                  if p6_mode in ('per_image', 'batch') else None

        # P7: ScaledResidual on GCN branch
        self.p7 = ScaledResidual(init_alpha=p7_alpha) if p7_mode == 'scaled' else None

    def _knn(self, x):
        B, C, H, W = x.shape
        N = H * W
        ke = min(self.k, N - 1)
        if ke == 0:
            return torch.empty((2, 0), dtype=torch.long, device=x.device), B, N, 0
        with torch.amp.autocast('cuda', enabled=False):
            xf = x.permute(0, 2, 3, 1).reshape(B, N, C).float()
            xn = F.normalize(xf, dim=-1, eps=1e-6)
            sim = torch.bmm(xn, xn.transpose(1, 2))
            sim.diagonal(dim1=1, dim2=2).fill_(-float('inf'))
            _, idx = sim.topk(ke, dim=-1)
        base = torch.arange(B, device=x.device).view(B, 1, 1) * N
        row = (torch.arange(N, device=x.device).view(1, N, 1) + base).expand(B, N, ke)
        col = idx + base.expand(B, N, ke)
        ei = torch.stack([row.reshape(-1), col.reshape(-1)], 0)
        return ei, B, N, ke

    def forward(self, x):
        B, C, H, W = x.shape
        h = self.norm1(x)
        ei, Bg, N, ke = self._knn(h)
        h_flat = h.permute(0, 2, 3, 1).reshape(Bg * N, C)
        h_conv = self.gcn(h_flat, ei, ke)
        if self.p6 is not None:
            h_conv = self.p6(h_conv, Bg, N)
        h_conv = h_conv.reshape(Bg, H, W, C).permute(0, 3, 1, 2)

        if self.p7 is not None:
            x = self.p7(x, self.drop(h_conv))
        else:
            x = x + self.drop(h_conv)

        h = self.norm2(x)
        x = x + self.drop(self.ffn(h))
        return x


class PyramidViGP2(nn.Module):
    def __init__(self, num_classes=1000, embed_dim=48, depths=(2, 2, 6, 2),
                 drop_path_rate=0.1, k=9,
                 p6_mode=None, p6_groups=8,
                 p7_mode=None, p7_alpha=0.3):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, embed_dim, 16, 16, bias=False),
            LayerNorm2d(embed_dim), nn.GELU(),
        )
        self.stages = nn.ModuleList()
        dpr = [x.item() for x in torch.linspace(0, drop_path_rate, sum(depths))]
        cur = 0
        for i, d in enumerate(depths):
            dim = embed_dim * (2 ** i)
            ds = nn.Sequential(
                nn.Conv2d(dim // 2, dim, 2, 2, bias=False),
                LayerNorm2d(dim),
            ) if i > 0 else nn.Identity()
            stage = nn.Sequential(
                ds,
                *[GrapherP2(dim, k=k, drop_path=dpr[cur + j],
                            p6_mode=p6_mode, p6_groups=p6_groups,
                            p7_mode=p7_mode, p7_alpha=p7_alpha)
                  for j in range(d)]
            )
            self.stages.append(stage)
            cur += d
        self.norm = nn.LayerNorm(embed_dim * 8)
        self.head = nn.Linear(embed_dim * 8, num_classes)
        self.apply(self._init)

    def _init(self, m):
        if isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=.02)
            if m.bias is not None: nn.init.constant_(m.bias, 0)
        elif isinstance(m, (nn.LayerNorm, LayerNorm2d)):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)

    def forward(self, x):
        x = self.stem(x)
        for s in self.stages: x = s(x)
        x = F.adaptive_avg_pool2d(x, 1).flatten(1)
        return self.head(self.norm(x))


def build_p2_model(total_blocks=12, k=9, num_classes=100,
                   p6_mode=None, p6_groups=8,
                   p7_mode=None, p7_alpha=0.3):
    depths = make_depths(total_blocks)
    m = PyramidViGP2(num_classes=num_classes, embed_dim=48,
                     depths=depths, drop_path_rate=0.1, k=k,
                     p6_mode=p6_mode, p6_groups=p6_groups,
                     p7_mode=p7_mode, p7_alpha=p7_alpha)
    return m, depths


# ---- Sanity check: build one of each variant and print params ----
print("Sanity check of Phase 2A model builders:")
for label, kwargs in [
    ("P6a per_image", dict(p6_mode='per_image')),
    ("P6b batch",     dict(p6_mode='batch')),
    ("P7a scaled",    dict(p7_mode='scaled')),
]:
    m, d = build_p2_model(16, k=9, num_classes=NUM_CLASSES, **kwargs)
    n = sum(p.numel() for p in m.parameters())
    print(f"  {label:16s} | depths={d} | params={n/1e6:.2f}M")
    del m; torch.cuda.empty_cache()
print("✓ All Phase 2A variants build successfully.")

Sanity check of Phase 2A model builders:
  P6a per_image    | depths=(3, 3, 8, 2) | params=6.73M
  P6b batch        | depths=(3, 3, 8, 2) | params=6.73M
  P7a scaled       | depths=(3, 3, 8, 2) | params=6.72M
✓ All Phase 2A variants build successfully.


In [37]:
# =====================================================================
# Cell 5 — Phase 2A training / evaluation
# =====================================================================
def train_epoch_p2(model, loader, opt, sc, crit, device):
    model.train(); tl = 0.0; n = 0
    for imgs, tgts in tqdm(loader, desc="Train", leave=False):
        imgs = imgs.to(device, non_blocking=True)
        tgts = tgts.to(device, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with autocast('cuda'):
            loss = crit(model(imgs), tgts)
        if torch.isnan(loss) or torch.isinf(loss):
            opt.zero_grad(set_to_none=True); continue
        sc.scale(loss).backward()
        sc.unscale_(opt)
        for p in model.parameters():
            if p.grad is not None:
                torch.nan_to_num_(p.grad, nan=0.0, posinf=0.0, neginf=0.0)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        sc.step(opt); sc.update()
        tl += loss.item() * imgs.size(0); n += imgs.size(0)
    return tl / max(n, 1)


@torch.no_grad()
def validate_p2(model, loader, crit, device):
    model.eval(); tl = c = n = 0
    for imgs, tgts in tqdm(loader, desc="Val", leave=False):
        imgs = imgs.to(device, non_blocking=True)
        tgts = tgts.to(device, non_blocking=True)
        with autocast('cuda'):
            out = model(imgs)
            tl += crit(out, tgts).item() * imgs.size(0)
        c += (out.argmax(1) == tgts).sum().item(); n += tgts.size(0)
    return tl / n, c / n


def run_p2_pilot(tag, build_fn, epochs=P2_EPOCHS):
    """Train one Phase 2 config; return (best_acc, history)."""
    print("=" * 80); print(f"PILOT [{tag}]"); print("=" * 80)
    model, depths = build_fn()
    n_params = sum(p.numel() for p in model.parameters())
    print(f"  depths={depths} | params={n_params/1e6:.2f}M")
    model = model.to(DEVICE)

    tr_c = nn.CrossEntropyLoss(); vl_c = nn.CrossEntropyLoss()
    decay, no_decay = [], []
    for n, p in model.named_parameters():
        if p.ndim <= 1 or 'norm' in n.lower() or 'bias' in n.lower():
            no_decay.append(p)
        else:
            decay.append(p)
    opt = optim.AdamW([{'params': decay, 'weight_decay': P2_WD},
                       {'params': no_decay, 'weight_decay': 0.0}], lr=P2_LR)
    sc  = GradScaler('cuda')
    sch = CosineLRScheduler(opt, t_initial=epochs, lr_min=1e-6,
                            warmup_t=P2_WARMUP, warmup_lr_init=1e-6)
    sch.step(0)

    best, hist = 0.0, []
    for ep in range(epochs):
        t0 = time.time()
        tl = train_epoch_p2(model, train_loader, opt, sc, tr_c, DEVICE)
        vl, va = validate_p2(model, val_loader, vl_c, DEVICE)
        sch.step(ep + 1); dt = time.time() - t0
        hist.append(dict(tag=tag, epoch=ep + 1, train_loss=tl,
                         val_loss=vl, val_acc=va, time=dt))
        star = "★" if va > best else " "
        if va > best: best = va
        print(f"  Ep {ep+1:2d}/{epochs} | TrL {tl:.4f} | VaL {vl:.4f} | "
              f"Acc {va:.4f} | {dt:.1f}s {star}")
    print(f"  → Best: {best:.4f} | vs baseline "
          f"{BASELINE_MEAN:.4f} (delta {best-BASELINE_MEAN:+.4f})\n")
    del model, opt, sc; torch.cuda.empty_cache()
    return best, hist

In [38]:
# =====================================================================
# Cell 6 — Run Phase 2A pilots
# =====================================================================
P2A_RESULTS = []
P2A_HISTORY = []

# --- P6-a: DiffGroupNorm per-image at depth 16 ---
best, hist = run_p2_pilot(
    "P6a_diffGN_perimage_d16",
    lambda: build_p2_model(16, k=9, num_classes=NUM_CLASSES,
                           p6_mode='per_image', p6_groups=8)
)
P2A_RESULTS.append(dict(exp="P6a", method="DiffGN-per-image",
                        blocks=16, best_acc=best))
P2A_HISTORY.extend(hist)

# --- P6-b: DiffGroupNorm batch-style at depth 16 ---
best, hist = run_p2_pilot(
    "P6b_diffGN_batch_d16",
    lambda: build_p2_model(16, k=9, num_classes=NUM_CLASSES,
                           p6_mode='batch', p6_groups=8)
)
P2A_RESULTS.append(dict(exp="P6b", method="DiffGN-batch",
                        blocks=16, best_acc=best))
P2A_HISTORY.extend(hist)

# --- P7-a: ScaledResidual at depth 16 ---
best, hist = run_p2_pilot(
    "P7a_scaledRes_d16",
    lambda: build_p2_model(16, k=9, num_classes=NUM_CLASSES,
                           p7_alpha=0.3)
)
P2A_RESULTS.append(dict(exp="P7a", method="ScaledResidual",
                        blocks=16, best_acc=best))
P2A_HISTORY.extend(hist)

# --- P7-b: ScaledResidual at depth 20 ---
best, hist = run_p2_pilot(
    "P7b_scaledRes_d20",
    lambda: build_p2_model(20, k=9, num_classes=NUM_CLASSES,
                           p7_alpha=0.3)
)
P2A_RESULTS.append(dict(exp="P7b", method="ScaledResidual",
                        blocks=20, best_acc=best))
P2A_HISTORY.extend(hist)

df_p2a = pd.DataFrame(P2A_RESULTS)
df_p2a["delta_vs_baseline"] = df_p2a["best_acc"] - BASELINE_MEAN
df_p2a["significant"] = df_p2a["delta_vs_baseline"].abs() > BASELINE_STD
print("\n" + "=" * 80)
print("PHASE 2A RESULTS")
print("=" * 80)
print(df_p2a.to_string(index=False))
df_p2a.to_csv(os.path.join(PHASE2_DIR, "phase2a_results.csv"), index=False)

PILOT [P6a_diffGN_perimage_d16]
  depths=(3, 3, 8, 2) | params=6.73M


  Ep  1/30 | TrL 4.6449 | VaL 4.6092 | Acc 0.0112 | 73.1s ★


  Ep  2/30 | TrL 4.3620 | VaL 3.9165 | Acc 0.1092 | 73.6s ★


  Ep  3/30 | TrL 4.1438 | VaL 3.6420 | Acc 0.1434 | 74.4s ★


  Ep  4/30 | TrL 4.0021 | VaL 3.5343 | Acc 0.1648 | 72.4s ★


  Ep  5/30 | TrL 3.8963 | VaL 3.4378 | Acc 0.1872 | 74.4s ★


... (110 outputs left out)

  Ep 27/30 | TrL 2.7327 | VaL 2.2144 | Acc 0.4368 | 69.6s ★


  Ep 28/30 | TrL 2.7246 | VaL 2.2018 | Acc 0.4440 | 69.8s ★


  Ep 29/30 | TrL 2.7157 | VaL 2.1943 | Acc 0.4376 | 69.7s  


  Ep 30/30 | TrL 2.7082 | VaL 2.1890 | Acc 0.4424 | 70.1s  
  → Best: 0.4440 | vs baseline 0.4520 (delta -0.0080)


PHASE 2A RESULTS
exp           method  blocks  best_acc  delta_vs_baseline  significant
P6a DiffGN-per-image      16    0.4648             0.0128         True
P6b     DiffGN-batch      16    0.4624             0.0104         True
P7a   ScaledResidual      16    0.4484            -0.0036        False
P7b   ScaledResidual      20    0.4440            -0.0080         True
